In [1]:
# ==========================================================
# DAY 17 - VENDOR RELIABILITY METRICS
# ==========================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

INPUT_FILE = PROJECT_ROOT / "03_processed_data/master_vendor_dataset.csv"
OUTPUT_FILE = PROJECT_ROOT / "03_processed_data/vendor_reliability.csv"

df = pd.read_csv(INPUT_FILE)

print("Loaded:", df.shape)

# Reliability = Low defect + Stable cost
df["reliability_score"] = (
    (1 - df["defect_score"]) * 0.6 +
    (1 - df["cost_score"]) * 0.4
)

# Categorize vendors
def classify(score):
    if score >= 0.7:
        return "Highly Reliable"
    elif score >= 0.5:
        return "Moderately Reliable"
    else:
        return "Unreliable"

df["reliability_category"] = df["reliability_score"].apply(classify)

# ----------------------------------------------------------
# RISK SCORE
# ----------------------------------------------------------

df["risk_score"] = (
    df["defect_score"] * 0.7 +
    df["cost_score"] * 0.3
)

# Risk category
def risk_label(x):
    if x > 0.7:
        return "High Risk"
    elif x > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_category"] = df["risk_score"].apply(risk_label)

# Stability (variance proxy using z-score)
df["stability"] = 1 / (1 + abs(df["cost_zscore"]))

# Combined reliability (improved)
df["reliability_score"] = (
    (1 - df["risk_score"]) * 0.6 +
    df["stability"] * 0.4
)

df.to_csv(OUTPUT_FILE, index=False)

Loaded: (20, 14)
